# Parse House Roll Call PDF files

This notebook parses and saves data to CSV files from the House Roll Call vote PDFs that are stored in ```../../data/HJRollCalls'``` subfolders.

## Parsing a Single PDF File

First, I will parse the first House Roll Call Vote pdf in our dataset (stored at ```../../data/HJRollCalls/2003/hj01082003.pdf```)

This approach uses PyMuPDF to extract text from the PDF by line, then applies regular expressions to appropriately capture the data. This method works well because we know that the structure of the House Roll Call Vote Journals is consistent across the years, so regular expressions can capture the names of the different representatives as long as they are robust enough to handle some varied formatting.

### Setup

In [1]:
import pymupdf
import re
import pandas as pd
from pathlib import Path

Regular expressions needed to parse data are defined.

Depending on the year of the document, the required Regular expressions vary.

In [2]:
# Define regular expressions for validating a file's content
pre2009_patterns_to_match =[
    '^\s+MASSACHUSETTS HOUSE OF REPRESENTATIVES\s+$',
    '^\s+Yea and Nay\s+(?P<RCDate>\d{2}/\d{2}/\d{2})\s(?P<RCTime>\d{2}:\d{2})\s\D$',
    '^\s+No.\s+(?P<RCNumberForYear>\d+)\s+(?P<YeaVoteCount>\d+)\s+YEAS\s+(?P<NayVoteCount>\d+)\s+NAYS\s+(?P<NotVotingCount>\d+)\s+N/V\s+$',
    '^\s+(?P<Vote1>[Y,N,X,P])\s+(?P<Name1>Mr. Speaker|(.+?)(?:\s{,1}[A-Za-z.,\'-]+)+)\s+(?P<Vote2>[Y,N,X,P])\s+(?P<Name2>(.+?)(?:\s{,1}[A-Za-z.,\'-]+)+)\s+(?P<Vote3>[Y,N,X,P])\s+(?P<Name3>(.+?)(?:\s{,1}[A-Za-z.,\'-]+)+)\s+((?P<Vote4>[Y,N,X,P])\s+(?P<Name4>(.+?)(?:\s{,1}[A-Za-z.,\'-]+)+)\s+)?$',
    '^\s+\*=AFTER VOTE\s+P=PRESENT\s+X=NOT VOTING\s+$'
]

In [3]:
post2017_patterns_to_match = [
    '^(?P<RCDate>\d{2}/\d{2}/\d{4})\s(?P<RCTime>\d{2}:\d{2} \D{2})$',
    '^\s+MASSACHUSETTS HOUSE OF REPRESENTATIVES$',
    '^(?P<VoteCount>\d+)\s(?P<VoteType>\D+)$',
    '^No.\s+(?P<RCNumberForYear>\d+)$',
    '^Yea and Nay$',
    '^(?P<Name>Mr. Speaker|([A-Z-][À-ÿa-z.,\'’-]+?)(?: {,1}[\*À-ÿA-Za-z.,\'’-]+)+)$',
    '^(?P<VoteValue>\D{1})$',
    '^\*=AFTER VOTE\s+P=PRESENT\s+X=NOT VOTING$'
]

There are a few files that specifically do not match the patterns seen in many of the others. It appears that although the document appears the same way compared to files from after 2017, the files from 2023 and onwards fix some of the encoding issues that resulted in Nay votes appearing out of order and the main title and other features of the document not appearing in expected order.

While a more robust solution can be expanded (or split into a separate function) for simplicity for now, the functionality is being controlled by tracking the years to except and will be updated in the future during clean-up phase of this project

In [23]:
years_to_except = [2023, 2024, 2025]

Parsing functions for the different years of PDFs are defined. Currently PDFs from before 2009 and after 2017 are supported and are defined in the following two functions.

In [32]:
def parse_Pre2009_PDF(path, bill_id, verbose=False):

    doc = pymupdf.open(path)
    all_pdf_bills = []
    all_pdf_votes = []

    for page in doc:
        text = page.get_text()
        lines = text.split('\n')

        current_pdf_bill_data = []
        current_pdf_bill_data.append(bill_id)

        current_pdf_vote_data = []

        for i, line in enumerate(lines):
            if verbose: print(f'{i}: {line}')

            # Skip parsing if blank line
            if line.isspace():
                if verbose: print("Found blank line")
                continue

            if i == 2: # parse full title line individually for ease
                current_pdf_bill_data.append(line.strip())

            # match = re.search(title_regex, line)
            matched = False
            for j, pattern in enumerate(pre2009_patterns_to_match):
                match = re.match(pattern, line)

                if match:
                    matched = True
                    if verbose: print(f'Matched with pattern {pattern}')

                    if j == 0:
                        # Do nothing, just the title
                        if verbose: print("Parsing title line")
                    elif j == 1:
                        # Extract vote date and time
                        rc_date = match.group('RCDate')
                        current_pdf_bill_data.append(rc_date)
                        rc_time = match.group('RCTime')
                        current_pdf_bill_data.append(rc_time)
                        if verbose: print(f"Found date: {rc_date} and time: {rc_time}")
                    elif j == 2:
                        rc_count_for_year = match.group('RCNumberForYear')
                        current_pdf_bill_data.append(rc_count_for_year)

                        rc_yea_vote_count = match.group('YeaVoteCount')
                        current_pdf_bill_data.append(rc_yea_vote_count)

                        rc_nay_vote_count = match.group('NayVoteCount')
                        current_pdf_bill_data.append(rc_nay_vote_count)

                        rc_not_voting_count = match.group('NotVotingCount')
                        current_pdf_bill_data.append(rc_not_voting_count)

                        if verbose: print(f'Found {rc_yea_vote_count} YEA votes, {rc_nay_vote_count} Nay votes, {rc_not_voting_count} NV votes for RC Vote {rc_count_for_year}')
                    elif j == 3:
                        vote1 = match.group('Vote1')
                        name1 = match.group('Name1')
                        current_pdf_vote_data.append([bill_id, name1, vote1])

                        vote2 = match.group('Vote2')
                        name2 = match.group('Name2')
                        current_pdf_vote_data.append([bill_id, name2, vote2])

                        vote3 = match.group('Vote3')
                        name3 = match.group('Name3')
                        current_pdf_vote_data.append([bill_id, name3, vote3])

                        vote4 = match.group('Vote4')
                        name4 = match.group('Name4')
                        current_pdf_vote_data.append([bill_id, name4, vote4])

                        if verbose: print(f'Found {name1} with vote {vote1}, {name2} with vote {vote2}, {name3} with {vote3}')
                    elif j == 4:
                        if verbose: print("Parsing end line")

            if not matched:
                Exception(f"Failed to match regex on line {i} : {line}")

        all_pdf_bills.append(current_pdf_bill_data)
        all_pdf_votes.extend(current_pdf_vote_data)
        bill_id += 1

    new_bill_id = bill_id
    print(f"Succesfully parsed file at path: {path}")
    if verbose:
        print(f'Found {len(all_pdf_votes)} voting records')
        print(f'Found {len(all_pdf_bills)} bill records')
    return all_pdf_bills, all_pdf_votes, new_bill_id

In [54]:
def parse_Post2017_PDF(path, bill_id, verbose=False):

    doc = pymupdf.open(path)
    all_pdf_bills = []
    all_pdf_votes = []

    file_year = int(str(path)[23:27])

    for page in doc:
        text = page.get_text()
        lines = text.split('\n')

        current_pdf_bill_data = []
        current_pdf_bill_data.append(bill_id)

        current_pdf_vote_data = []
        nay_vote_found = False
        end_found = False
        for i, line in enumerate(lines):
            if verbose: print(f'{i}: {line}')

            # Skip parsing if blank line
            if line.isspace():
                if verbose: print("Found blank line")
                continue

            if i == 7 and file_year not in years_to_except: # parse full title line individually for ease
                current_pdf_bill_data.insert(1, line.strip())
            elif file_year in years_to_except and i == 1:
                current_pdf_bill_data.insert(1, line.strip())

            # match = re.search(title_regex, line)
            matched = False
            if not end_found:
                for j, pattern in enumerate(post2017_patterns_to_match):
                    match = re.match(pattern, line)

                    if match:
                        matched = True
                        if verbose: print(f'Matched with pattern {pattern}')

                        if j == 0:
                            # Extract vote date and time
                            rc_date = match.group('RCDate')
                            current_pdf_bill_data.append(rc_date)
                            rc_time = match.group('RCTime')
                            current_pdf_bill_data.append(rc_time)
                            if verbose: print(f"Found date: {rc_date} and time: {rc_time}")
                        elif j == 1:
                            # Do nothing, just the title
                            if verbose: print("Parsing title line")
                            break
                        elif j == 2:
                            # Get vote count and type of vote since they appear each on a new line
                            rc_vote_count = match.group('VoteCount')
                            current_pdf_bill_data.append(rc_vote_count)

                            rc_vote_type = match.group('VoteType')

                            if verbose: print(f'Found {rc_vote_count} {rc_vote_type} votes')
                        elif j == 3:
                            # Parse RC Vote number
                            rc_count_for_year = match.group('RCNumberForYear')
                            current_pdf_bill_data.insert(3, rc_count_for_year) # Insert at index so it matches format of other PDF styles


                            if verbose: print(f'Found No. {rc_count_for_year} on current line')
                        elif j == 4:
                            # Parsing Yays and Nays text, do nothing
                            if verbose: print("Parsing \'Yays and Nays\'")
                            break
                        elif j == 5:
                            # Skip parsing this line if previous line was a Nay Vote
                            if nay_vote_found and file_year not in years_to_except:
                                nay_vote_found = False
                                print("Turning off nay vote found switch")
                                continue

                            # Parsing Voter Names
                            name = match.group('Name')

                            # Get Next line from list of lines to parse name for
                            vote_match = re.match('^(?P<VoteValue>\D{1})$', lines[i+1])
                            vote = vote_match.group('VoteValue')

                            current_pdf_vote_data.append([bill_id, name, vote])

                            if verbose: print(f'Found vote {vote} from Rep. {name}')
                        elif j == 6:
                            # Parse vote value, if value is N for Nay, parse name as well since they appear in reverse order

                            # Get Next line from list of lines to parse name for
                            vote_match = re.match('^(?P<VoteValue>\D{1})$', lines[i])
                            vote = vote_match.group('VoteValue')

                            if vote == 'N' and file_year not in years_to_except:
                                name_match = re.match('^(?P<Name>Mr. Speaker|([A-Z-][À-ÿa-z.,\'’-]+?)(?: {,1}[\*À-ÿA-Za-z.,\'’-]+)+)$', lines[i+1])
                                name = name_match.group('Name')

                                current_pdf_vote_data.append([bill_id, name, vote])
                                nay_vote_found = True
                                if verbose: print("Found Nay vote, will skip parsing next name")
                            else:
                                continue
                        elif j == 7:
                            if verbose: print("Parsing end line")
                            end_found = True
                            break
                if not matched:
                    Exception(f"Failed to match regex on line {i} : {line}")

        all_pdf_bills.append(current_pdf_bill_data)
        all_pdf_votes.extend(current_pdf_vote_data)
        bill_id += 1

    new_bill_id = bill_id
    print(f"Succesfully parsed file at path: {path}")
    if verbose:
        print(f'Found {len(all_pdf_votes)} voting records')
        print(f'Found {len(all_pdf_bills)} bill records')
    return all_pdf_bills, all_pdf_votes, new_bill_id

### Testing on a Single PDF document

In [15]:
bill_data = []
vote_data = []
bill_id = 0
path = "../../data/HJRollCalls/2003/hj01082003.pdf"

# Call defined function to get data for current pdf
pdf_bill_data, pdf_vote_data, bill_id = parse_Pre2009_PDF(path, bill_id)

bill_data.extend(pdf_bill_data)
vote_data.extend(pdf_vote_data)

Succesfully parsed file at path: ../../data/HJRollCalls/2003/hj01082003.pdf


In [71]:
vote_data

[[0, 'Mr. Speaker', 'Y'],
 [0, 'Fresolo', 'Y'],
 [0, 'Murphy, K.J.', 'Y'],
 [0, 'Swan', 'Y'],
 [0, 'Asselin', 'Y'],
 [0, 'Galvin', 'Y'],
 [0, 'Nangle', 'Y'],
 [0, 'Teahan', 'Y'],
 [0, 'Atkins', 'Y'],
 [0, 'Garry', 'Y'],
 [0, 'Naughton', 'Y'],
 [0, 'Timilty', 'Y'],
 [0, 'Atsalis', 'Y'],
 [0, 'Gobi', 'X'],
 [0, 'Nyman', 'Y'],
 [0, 'Tobin', 'Y'],
 [0, 'Ayers', 'Y'],
 [0, 'Goguen', 'Y'],
 [0, "O'Brien", 'Y'],
 [0, 'Toomey', 'Y'],
 [0, 'Balser', 'Y'],
 [0, 'Golden B.P.', 'Y'],
 [0, "O'Flaherty", 'Y'],
 [0, 'Torrisi', 'Y'],
 [0, 'Binienda', 'Y'],
 [0, 'Golden T.A.', 'Y'],
 [0, 'Owens-Hicks', 'Y'],
 [0, 'Travis', 'Y'],
 [0, 'Blumer', 'Y'],
 [0, 'Grant', 'Y'],
 [0, 'Parente', 'Y'],
 [0, 'Turkington', 'Y'],
 [0, 'Bosley', 'Y'],
 [0, 'Greene', 'Y'],
 [0, 'Patrick', 'Y'],
 [0, 'Vallee', 'X'],
 [0, 'Bradley', 'Y'],
 [0, 'Haddad', 'Y'],
 [0, 'Paulsen', 'Y'],
 [0, 'Verga', 'X'],
 [0, 'Broadhurst', 'Y'],
 [0, 'Hall', 'Y'],
 [0, 'Pedone', 'Y'],
 [0, 'Wagner', 'Y'],
 [0, 'Buoniconti', 'Y'],
 [0, 'Harki

In [72]:
bill_data

[[0, 'QUORUM', '01/08/03', '01:44', '1', '151', '0', '9'],
 [1,
  'H2003  On motion to       Postpone',
  '01/08/03',
  '02:01',
  '2',
  '45',
  '107',
  '8'],
 [2, 'H2003  On Amendment', '01/08/03', '02:20', '3', '41', '110', '9'],
 [3, 'H2003  On Amendment', '01/08/03', '02:45', '4', '44', '107', '9'],
 [4, 'H2003  On Amendment', '01/08/03', '04:03', '5', '28', '119', '13'],
 [5, 'H2003  On Amendment', '01/08/03', '04:34', '6', '22', '128', '10']]

In [73]:
# Create a dataframe from the parsed PDF data for bills
bill_df = pd.DataFrame(bill_data, columns=['bill_id', 'bill_name', 'rollcall_date', 'rollcall_time', 'rollcall_count_for_year', 'yea_vote_count', 'nay_vote_count', 'not_voting_count'])

bill_df.set_index('bill_id', inplace=True)
bill_df

,bill_name,rollcall_date,rollcall_time,rollcall_count_for_year,yea_vote_count,nay_vote_count,not_voting_count
bill_id,,,,,,,
0,QUORUM,01/08/03,01:44,1,151,0,9
1,H2003 On motion to Postpone,01/08/03,02:01,2,45,107,8
2,H2003 On Amendment,01/08/03,02:20,3,41,110,9
3,H2003 On Amendment,01/08/03,02:45,4,44,107,9
4,H2003 On Amendment,01/08/03,04:03,5,28,119,13
5,H2003 On Amendment,01/08/03,04:34,6,22,128,10


In [74]:
# Create a dataframe from parsed PDF data about votes
vote_df = pd.DataFrame(vote_data, columns=['bill_id', 'voter_name', 'vote_value'])
vote_df

,bill_id,voter_name,vote_value
0,0,Mr. Speaker,Y
1,0,Fresolo,Y
2,0,"Murphy, K.J.",Y
3,0,Swan,Y
4,0,Asselin,Y
...,...,...,...
955,5,Webster,Y
956,5,Fox,N
957,5,"Murphy, J.M.",N
958,5,"Sullivan, J.C.",N


## Batch Processing all PDFs

Now that we have a function for scraping the data from all pages of a single pdf, we can loop through all of the pdfs that are stored on disk in the data and create the full dataset

In [41]:
# Get path to all pdfs stored in HJRollCall folder
all_paths = []
root_path = Path('../../data/HJRollCalls')

for path in root_path.glob('**/*.pdf'):
    all_paths.append(path)

print(len(all_paths))

599


### Parse all PDFs

This block is the core part of this notebook that actually iterates over the whole folder of PDFs and parses them.

Currently, this function is set to parse roll call votes that are either < 2009 or 2017 >. This is because PDFs from the years 2009 - 2017 are done is a specific style where the actual votes are stored as an image embedded in the PDF, not text, and require some special attention. In the worst case, this data may need to be manually scraped but future experiementation will be done to see if we can automate this process.

Each parse_####_PDF() function call takes 3 variables, path = the file path to parse, bill_id = the iterator of all bills parsed (used for reference in the database table), and an optional named parameter verbose, which is set to False by default. When verbose is set to True, the function will output debugging messages in case something goes wrong during the parsing process.

In [59]:
bill_id = -1
bill_data = []
vote_data = []

for path in all_paths:

    # If path includes the year <2009, skip for now
    file_year = int(str(path)[23:27])
    print(f'Attempting to parsing file at path: {path}')
    if file_year < 2009:
        current_bill_data, current_vote_data, bill_id = parse_Pre2009_PDF(path, bill_id+1)
    # Skip files from 2009 - 2017 for now since they will have more things to deal with for parsing
    elif file_year >= 2009 and file_year <= 2017:
        print(f'Skipping file from year: {file_year}')
        continue
    elif file_year > 2017:
        current_bill_data, current_vote_data, bill_id = parse_Post2017_PDF(path, bill_id+1)
    else:
        print(f"File in unexpected year attempted to be parsed: {path, file_year}")
        break

    bill_data.extend(current_bill_data)
    vote_data.extend(current_vote_data)

Attempting to parsing file at path: ../../data/HJRollCalls/2013/hj04032013.pdf
Skipping file from year: 2013
Attempting to parsing file at path: ../../data/HJRollCalls/2013/hj07172013.pdf
Skipping file from year: 2013
Attempting to parsing file at path: ../../data/HJRollCalls/2013/hj09252013.pdf
Skipping file from year: 2013
Attempting to parsing file at path: ../../data/HJRollCalls/2013/hj11182013.pdf
Skipping file from year: 2013
Attempting to parsing file at path: ../../data/HJRollCalls/2013/hj10022013.pdf
Skipping file from year: 2013
Attempting to parsing file at path: ../../data/HJRollCalls/2013/hj06052013.pdf
Skipping file from year: 2013
Attempting to parsing file at path: ../../data/HJRollCalls/2013/hj07302013.pdf
Skipping file from year: 2013
Attempting to parsing file at path: ../../data/HJRollCalls/2013/hj01232013.pdf
Skipping file from year: 2013
Attempting to parsing file at path: ../../data/HJRollCalls/2013/hj11132013.pdf
Skipping file from year: 2013
Attempting to parsi

### Creating and writing DataFrames from parsed data then converting to CSV

In [60]:
bill_data[0]

[0, 'QUORUM', '06/08/2022', '01:36 PM', '194', '155', '0', '0']

In [61]:
# Create a dataframe from the parsed PDF data for bills
bill_df = pd.DataFrame(bill_data, columns=['bill_id', 'bill_name', 'rollcall_date', 'rollcall_time', 'rollcall_count_for_year', 'yea_vote_count', 'nay_vote_count', 'not_voting_count'])

bill_df.set_index('bill_id', inplace=True)
bill_df

,bill_name,rollcall_date,rollcall_time,rollcall_count_for_year,yea_vote_count,nay_vote_count,not_voting_count
bill_id,,,,,,,
0,QUORUM,06/08/2022,01:36 PM,194,155,0,0
1,H. 4805 shall this pass notwithstanding Objection,06/08/2022,01:50 PM,195,119,36,0
3,QUORUM,07/13/2022,01:32 PM,212,154,0,0
4,"H. 5007 Consolidated Amendment ""A""",07/13/2022,05:49 PM,213,154,0,0
6,QUORUM,02/16/2022,01:50 PM,144,156,0,0
...,...,...,...,...,...,...,...
3567,H. 3899 On Enactment Land Taking,09/30/03,02:14,416,147,0,13
3568,QUORUM,09/30/03,02:55,417,150,0,10
3569,S. 187 On Engrossment,09/30/03,03:10,418,146,4,10


In [62]:
# Create a dataframe from parsed PDF data about votes
vote_df = pd.DataFrame(vote_data, columns=['bill_id', 'voter_name', 'vote_value'])
vote_df

,bill_id,voter_name,vote_value
0,0,Mr. Speaker,Y
1,0,Farley-Bouvier,Y
2,0,Mahoney,Y
3,0,Turco,Y
4,0,Hogan,Y
...,...,...,...
508949,3571,Webster,N
508950,3571,Festa,Y
508951,3571,Miceli,Y
508952,3571,"Sullivan, D.B.",Y


The parsed output can be written to csv files for future cleaning and processing

In [66]:
bill_df.to_csv('../../data/parsedcsv/hjbills_raw.tsv', index=True, index_label=bill_df.index.name, sep='\t')
vote_df.to_csv('../../data/parsedcsv/hjrollcallvotes_raw.tsv', index=False, sep='\t')

Read the files from CSV to test that they wrote properly

In [68]:
bill_df_from_csv = pd.read_csv('../../data/parsedcsv/hjbills_raw.tsv', sep='\t')

bill_df_from_csv.head()

,bill_id,bill_name,rollcall_date,rollcall_time,rollcall_count_for_year,yea_vote_count,nay_vote_count,not_voting_count
0,0,QUORUM,06/08/2022,01:36 PM,194,155.0,0.0,0.0
1,1,H. 4805 shall this pass notwithstanding Objection,06/08/2022,01:50 PM,195,119.0,36.0,0.0
2,3,QUORUM,07/13/2022,01:32 PM,212,154.0,0.0,0.0
3,4,"H. 5007 Consolidated Amendment ""A""",07/13/2022,05:49 PM,213,154.0,0.0,0.0
4,6,QUORUM,02/16/2022,01:50 PM,144,156.0,0.0,0.0


In [70]:
vote_df_from_csv = pd.read_csv('../../data/parsedcsv/hjrollcallvotes_raw.tsv', sep='\t')
vote_df_from_csv.head()

,bill_id,voter_name,vote_value
0,0,Mr. Speaker,Y
1,0,Farley-Bouvier,Y
2,0,Mahoney,Y
3,0,Turco,Y
4,0,Hogan,Y


## Debugging individual PDFs

The following section is used for debugging specific documents that did not work. It is functionally identical to the section ### Testing on a single document, but is kept down here for reference for the work process, and less for presentation purposes.

In [55]:
path = '../../data/HJRollCalls/2019/hj09182019.pdf'
doc = pymupdf.open(path)

for page in doc:
    print("NEW DOC STARTING \n\n")
    text = page.get_text()
    lines = text.split('\n')

    for line in lines:
        print(line)


NEW DOC STARTING 


09/18/2019 02:17 PM
MASSACHUSETTS HOUSE OF REPRESENTATIVES
156 YEAS
1 NAYS
3 N/V
No. 91
Yea and Nay
H. 3854 notwithstanding the objection of the Gov
Mr. Speaker
Y
Farley-Bouvier
Y
McGonagle
Y
Vega
Y
Mariano
Y
Fernandes
Y
McMurtry
Y
Velis
Y
Haddad
Y
Ferrante
Y
Meschino
Y
Vincent
Y
Wagner *
Y
Finn
Y
Michlewitz
Y
Vitolo
Y
Donato
Y
Fiola
Y
Minicucci
Y
Walsh
Y
Moran M.
Y
Galvin
Y
Miranda
Y
Williams
Y
Arciero
Y
Garballey
Y
Mom
Y
Zlotnik
Y
Ashe
Y
Garlick
Y
Moran F.
Y
Jones
Y
Ayers
Y
Garry
Y
Murphy
Y
Hill
Y
Balser
Y
Gentile
Y
Murray
Y
Poirier
Y
Barber
Y
Golden
Y
Nangle
Y
Gifford
Y
Barrett
Y
Gonzalez
Y
Naughton
Y
Frost
Y
Benson
Y
Gordon
Y
Nguyen
Y
Barrows
Y
Biele
Y
Gouveia
Y
O’Day
Y
Berthiaume
Y
Blais
Y
Gregoire
Y
Parisella
Y
Boldyga
Y
Brodeur
Y
Haggerty
Y
Peake
Y
Crocker
Y
Cabral
Y
Hawkins
Y
Peisch
Y
D'Emilia
Y
Cahill
Y
Hay
Y
Petrolati
Y
DeCoste
Y
Campbell
Y
Hecht
Y
Pignatelli
X
N
Dooley
Capano
Y
Hendricks
Y
Provost
Y
Durant
Y
Carey
Y
Higgins
Y
Puppolo
Y
Ferguson
Y
Cassidy


In [56]:
bill_data, vote_data, bill_id = parse_Post2017_PDF(path, 0, verbose=True)

0: 09/18/2019 02:17 PM
Matched with pattern ^(?P<RCDate>\d{2}/\d{2}/\d{4})\s(?P<RCTime>\d{2}:\d{2} \D{2})$
Found date: 09/18/2019 and time: 02:17 PM
1: MASSACHUSETTS HOUSE OF REPRESENTATIVES
2: 156 YEAS
Matched with pattern ^(?P<VoteCount>\d+)\s(?P<VoteType>\D+)$
Found 156 YEAS votes
3: 1 NAYS
Matched with pattern ^(?P<VoteCount>\d+)\s(?P<VoteType>\D+)$
Found 1 NAYS votes
4: 3 N/V
Matched with pattern ^(?P<VoteCount>\d+)\s(?P<VoteType>\D+)$
Found 3 N/V votes
5: No. 91
Matched with pattern ^No.\s+(?P<RCNumberForYear>\d+)$
Found No. 91 on current line
6: Yea and Nay
Matched with pattern ^Yea and Nay$
Parsing 'Yays and Nays'
7: H. 3854 notwithstanding the objection of the Gov
8: Mr. Speaker
Matched with pattern ^(?P<Name>Mr. Speaker|([A-Z-][À-ÿa-z.,'’-]+?)(?: {,1}[\*À-ÿA-Za-z.,'’-]+)+)$
Found vote Y from Rep. Mr. Speaker
9: Y
Matched with pattern ^(?P<VoteValue>\D{1})$
10: Farley-Bouvier
Matched with pattern ^(?P<Name>Mr. Speaker|([A-Z-][À-ÿa-z.,'’-]+?)(?: {,1}[\*À-ÿA-Za-z.,'’-]+)+)$
Foun

In [57]:
# Create a dataframe from the parsed PDF data for bills
bill_df = pd.DataFrame(bill_data, columns=['bill_id', 'bill_name', 'rollcall_date', 'rollcall_time', 'rollcall_count_for_year', 'yea_vote_count', 'nay_vote_count', 'not_voting_count'])

bill_df.set_index('bill_id', inplace=True)
bill_df

,bill_name,rollcall_date,rollcall_time,rollcall_count_for_year,yea_vote_count,nay_vote_count,not_voting_count
bill_id,,,,,,,
0,H. 3854 notwithstanding the objection of the Gov,09/18/2019,02:17 PM,91,156,1,3
1,H. 4000 item 1595-6368 Transportation Trust Fund,09/18/2019,02:40 PM,92,125,31,4
2,H. 4000 item 2810-0100 Parks and Recreation,09/18/2019,02:45 PM,93,130,26,4
3,H. 4000 item 8900-0001 Department of Correction,09/18/2019,02:48 PM,94,125,31,4
4,H. 4076 On Engrossment,09/18/2019,03:13 PM,95,156,0,4


In [58]:
# Create a dataframe from parsed PDF data about votes
vote_df = pd.DataFrame(vote_data, columns=['bill_id', 'voter_name', 'vote_value'])
vote_df

,bill_id,voter_name,vote_value
0,0,Mr. Speaker,Y
1,0,Farley-Bouvier,Y
2,0,McGonagle,Y
3,0,Vega,Y
4,0,Mariano,Y
...,...,...,...
795,4,Wong,Y
796,4,Elugardo,Y
797,4,Markey,Y
798,4,Vargas,Y
